In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Any, TypedDict, Callable, Literal, cast
from shared import SharedCollection
import librosa
import pandas as pd

from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
import torch.nn as nn
import torch.optim as optim
import torch
from sklearn.preprocessing import StandardScaler

import matplotlib.pyplot as plt
import numpy as np
from functools import partial

tqdm.pandas()

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset
Audio is preprocessed on another machine, so we need to load them on this machine for training.

In [3]:
prefix = Path("../data")

In [4]:
with open(prefix.joinpath("labels.csv"), "r") as f:
    labels: pd.DataFrame = pd.read_csv(f)

Standardise path into prefix/raw/Type/file or prefix/processed/Type/file (discrepancy caused by moving dataset around machine)

In [5]:
labels.source_file = labels.source_file.progress_apply(lambda x: prefix.joinpath(Path("raw", *Path(x).parts[2:])))
labels.processed_file = labels.processed_file.progress_apply(
    lambda x: prefix.joinpath(Path("processed", *Path(x).parts[1:])))

  0%|          | 0/5940 [00:00<?, ?it/s]

  0%|          | 0/5940 [00:00<?, ?it/s]

In [6]:
labels[:3]

,source_file,processed_file,fire_seeker_gain,fire_seeker_low,fire_seeker_mid,fire_seeker_treble
0,data/raw/Gretsch/Gretsch_F_open_122_ID1_4.wav,data/processed/Gretsch/Gretsch_F_open_122_ID1_...,0.415212,0.382531,0.889004,0.717846
1,data/raw/Gretsch/Gretsch_F_open_122_ID1_4.wav,data/processed/Gretsch/Gretsch_F_open_122_ID1_...,0.828639,0.576327,0.966072,0.016324
2,data/raw/Gretsch/Gretsch_F_open_122_ID1_4.wav,data/processed/Gretsch/Gretsch_F_open_122_ID1_...,0.093933,0.267912,0.440884,0.377729


Filter out missing file

In [7]:
def filter_existing(row: pd.Series) -> bool:
    if row.processed_file.exists():
        return True
    else:
        print(f"\033[31mProcessed file {row.processed_file} not found, skipping\033[0m")
        return False

In [8]:
def calc_length(row: pd.Series) -> pd.Series:
    x, sr = librosa.load(row.processed_file)
    row["length"] = len(x)
    return row

In [9]:
tqdm.pandas(desc="Filtering")
filtered = labels[labels.progress_apply(filter_existing, axis=1)]
tqdm.pandas(desc="Calculating sample length")
filtered = filtered.progress_apply(calc_length, axis=1)

Filtering:   0%|          | 0/5940 [00:00<?, ?it/s]

Processed file data/processed/Gretsch/Gretsch_F_open_122_ID1_4_1.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_F_open_122_ID1_4_2.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_F_open_122_ID1_4_3.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_Am_open_222_ID1_10_1.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_Am_open_222_ID1_10_2.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_Am_open_222_ID1_10_3.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_B_open_022_ID1_6_1.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_B_open_022_ID1_6_2.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_B_open_022_ID1_6_3.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_Cm_open_122_ID1_9_1.wav not found, skipping
Processed file data/processed/Gretsch/Gretsch_Cm_open_122_ID1_9_2.wav not found, skipping
Processed fil

Calculating sample length:   0%|          | 0/4673 [00:00<?, ?it/s]

## Boilerplates
Scaffolding to share code across different implementations, for the ease of experimenting with various setting, e.g. different features or different models

In [ ]:
from shared import to_tensor

def extract_X_y(df: pd.DataFrame, *, X_labels: list[str],
                x_mapper: Callable[[pd.DataFrame], np.ndarray] = lambda x: x.to_numpy()) -> \
        tuple[np.ndarray, np.ndarray]:
    return x_mapper(df[X_labels]), df[
        ["fire_seeker_gain", "fire_seeker_low", "fire_seeker_mid", "fire_seeker_treble"]].to_numpy()


class Dataset[T: torch.Tensor | np.ndarray](TypedDict):
    X_train: T
    y_train: T
    X_val: T
    y_val: T
    X_test: T
    y_test: T

class Collection:
    def __init__(self, model: nn.Module,
                 loss: nn.modules.loss._Loss,
                 optimiser: Callable[[optim.optimizer.ParamsT], torch.optim.Optimizer],
                 scaler: StandardScaler):
        self.model = model.to(device)
        self.loss = loss.to(device)
        self.optimiser = optimiser(self.model.parameters())
        self.scaler = scaler

def to_shared_collection(c: Collection)->SharedCollection:
    return SharedCollection(c.model.to("cpu"), c.loss, c.scaler)
def to_device(dataset: Dataset[np.ndarray], d=device) -> Dataset[torch.Tensor]:
    return cast(Dataset, dict(
        map(lambda x: (x[0], to_tensor(x[1]).to(d)),
            dataset.items())))


def scale(dataset: Dataset[np.ndarray], scaler) -> Dataset[np.ndarray]:
    return Dataset(X_train=scaler.fit_transform(dataset["X_train"]), y_train=dataset["y_train"],
                   X_val=scaler.transform(dataset["X_val"]), y_val=dataset["y_val"],
                   X_test=scaler.transform(dataset["X_test"]), y_test=dataset["y_test"])


def split_dataset(X: np.ndarray, y: np.ndarray) -> Dataset[np.ndarray]:
    X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.2, random_state=0)
    X_val_unscaled, X_test_unscaled, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=0)
    return Dataset(X_train=X_train, y_train=y_train, X_val=X_val_unscaled, y_val=y_val, X_test=X_test_unscaled,
                   y_test=y_test)


def test_model(dataset: Dataset, c: Collection,
               sample: int | None = None, key: Literal["test", "val", "train"] = "test"):
    total = torch.tensor(0.0).to(device)
    n = len(dataset["X_" + key])
    tqdm.pandas(desc="Predicting")

    # slower than just throwing the entire tensor into model and loss, but makes writing the random sampling code easier
    for i in tqdm(range(n) if sample is None else np.random.choice(n, sample)):
        y_hat = c.model(dataset["X_" + key][i])
        l = c.loss(y_hat, dataset["y_" + key][i])
        print(f"y: {dataset["y_" + key][i]} ŷ: {y_hat} loss: {l}")
        total += l
    avg = total / n
    print(
        f"Average loss{" of entire dataset" if sample is not None else ""}: {avg if sample is None else c.loss(c.model(dataset["X_" + key]), dataset["y_" + key])}")


def train_model(c: Collection, dataset: Dataset[torch.Tensor],
                epochs: int) -> list[float]:
    loss_history = []
    for _ in (n := tqdm(range(epochs), desc="Training")):
        c.optimiser.zero_grad()
        output = c.model(dataset["X_train"])
        loss = c.loss(output, dataset["y_train"])
        loss.backward()
        c.optimiser.step()
        n.postfix = str(loss.item())
        loss_history.append(loss.item())
    return loss_history


def loss_history(hist):
    plt.title("Loss History")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.plot(hist)


## Naive implementation
For our first model, we'll only provide 3 features, root mean square, spectral centroid and zero crossing rate

In [ ]:
from shared.feat_ext import root_mean_square, spectral_centroid, zero_crossing_rate


def calculate_naive_features(row: pd.Series) -> pd.Series:
    x, sr = librosa.load(row.processed_file)

    row["root_mean_square"] = root_mean_square(x)
    row["spectral_centroid"] = spectral_centroid(x, sr=sr)
    row["zero_crossing_rate"] = zero_crossing_rate(x)
    return row

In [ ]:
tqdm.pandas(desc="Calculating Features")
naive_features = filtered.progress_apply(calculate_naive_features, axis=1)

In [ ]:
naive_X, naive_y = extract_X_y(naive_features, X_labels=["root_mean_square", "spectral_centroid", "zero_crossing_rate"])
naive_scaler = StandardScaler()
naive_dataset: Dataset = to_device(scale(split_dataset(naive_X, naive_y), naive_scaler))

We'll be using a basic MLP with only 4 layers (2 hidden) and all activation will be using RELU. Input layer will have our 3 only features, then it widens up to 70 then narrow back down until the output is of size 3 (the 3 parameters we're trying to predict).


In [ ]:

from shared.model import NaiveModel

In [ ]:
naive = Collection(
    NaiveModel(),
    nn.MSELoss(),
    partial(optim.SGD, lr=0.001, momentum=0.9),
    naive_scaler
)

naive_loss_history = train_model(naive, naive_dataset, 50000)

In [ ]:
loss_history(naive_loss_history)

In [ ]:
test_model(naive_dataset, naive, key="train", sample=3)
test_model(naive_dataset, naive, sample=3)

The naive attempts did not preform well for test dataset (and it definitely did not generalise well). With equally terrible accuracy for both seen and unseen data, this is expected, as we only have 3 features and it is very unlikely to be enough.

## Second attempt
For out second attempts, instead of calculating our features for the entire samples, we take slices of out sample and calculate features on them, hopefully this gives more information that the model need to preform better. Here, I kept the window and frame length as librosa's default.

In [ ]:
from librosa.feature import rms, spectral_centroid, zero_crossing_rate


def calculate_windowed_naive_features() -> Callable[[pd.Series], pd.Series]:
    samples = min(filtered[
                      "length"])  # truncate out excess, most audio clip should be ~5 second, but with some very small variation that we can just chop out

    def _inner(row: pd.Series) -> pd.Series:
        x, sr = librosa.load(row.processed_file)
        x = x[:samples]
        S = np.abs(librosa.stft(x))
        row["root_mean_square"] = rms(S=S)[0]
        row["spectral_centroid"] = spectral_centroid(y=x, sr=sr)[0]
        row["zero_crossing_rate"] = zero_crossing_rate(x)[0]
        return row

    return _inner

In [ ]:
tqdm.pandas(desc="Calculating Features")
windowed_naive_feature = filtered.progress_apply(calculate_windowed_naive_features(), axis=1)

Machine learning model expects a flat 1D array as features, so we have to concatenate the three feature column containing ndarry each into a single ndarry.

In [ ]:
def flatten(x: pd.DataFrame) -> np.ndarray:
    x = x.apply(
        lambda row: np.concatenate([row["root_mean_square"], row["spectral_centroid"], row["zero_crossing_rate"]],
                                   dtype=np.float32), axis=1)
    return np.vstack(x.to_numpy()).astype(np.float32)

In [ ]:
windowed_naive_X, windowed_naive_y = extract_X_y(windowed_naive_feature,
                                                 X_labels=["root_mean_square", "spectral_centroid",
                                                           "zero_crossing_rate"], x_mapper=flatten)
windowed_naive_dataset_scaler = StandardScaler()
windowed_naive_dataset = to_device(scale(split_dataset(windowed_naive_X, windowed_naive_y), windowed_naive_dataset_scaler))

I've decided to use the exact same model as before, but with a larger input layer to accommodate for all the features.

In [ ]:
windowed_naive_old_neurons = Collection(
    NaiveModel(input_n=windowed_naive_X.shape[1]),
    nn.MSELoss(),
    partial(optim.SGD, lr=0.001, momentum=0.9),
    windowed_naive_dataset_scaler
)

windowed_naive_old_neurons_loss_history = train_model(windowed_naive_old_neurons, windowed_naive_dataset, 50000)

In [ ]:
loss_history(windowed_naive_old_neurons_loss_history)

For comparison, I made a model with significantly more neuron on each hidden layer


In [ ]:
windowed_naive_larger_model = Collection(
    NaiveModel(input_n=windowed_naive_X.shape[1], h1_n=1000, h2_n=500, h3_n=100),
    nn.MSELoss(),
    partial(optim.SGD, lr=0.001, momentum=0.9),
    windowed_naive_dataset_scaler
)

windowed_naive_larger_model_loss_history = train_model(windowed_naive_larger_model, windowed_naive_dataset, 50000)

In [ ]:
loss_history(windowed_naive_larger_model_loss_history)

In [ ]:
test_model(windowed_naive_dataset, windowed_naive_old_neurons, key="train", sample=3)
test_model(windowed_naive_dataset, windowed_naive_old_neurons, key="val", sample=3)

In [ ]:
test_model(windowed_naive_dataset, windowed_naive_larger_model, key="train", sample=3)
test_model(windowed_naive_dataset, windowed_naive_larger_model, key="val", sample=3)

While both model preformed quite well at the training data, the one with same neurons count as our naive model preforms worse at training data compared to the larger model. On the validation dataset, both preform quite terribly, but the larger model produce slightly smaller loss.

Understandably, now that we slice and sample chunk of the sample, and have more than 3 features, the machine learning model have a lot more information to work with compared to the naive model. This is likely why the new model is so much better at training dataset than naive, but it seems like it did not generalise well, perhaps it simply learned to distinguish between all the audio sample in training dataset.

In [ ]:
print(f"{windowed_naive_X.shape[1]} features")
print(f"{windowed_naive_feature.spectral_centroid.iloc[0].shape[0]} value per features")

And in hindsight, I feel like a total of 636 features was excessive, I probably should've diverted from librosa's default, ~5 features per second and 25 features for an entire clips sound more reasonable.

## (A more) Sensible classical model
Instead of keeping all features over time, we've decided to calculate the mean _and the variance_ as features

In [ ]:

from shared.features import calculate_classical_features

In [ ]:
tqdm.pandas(desc="Calculating Features")
classical_feature = filtered.progress_apply(calculate_classical_features, axis=1)

In [ ]:
import shared
classical_X, classical_y = extract_X_y(classical_feature,
                                       X_labels=["root_mean_square_mean", "root_mean_square_variance",
                                                 "spectral_centroid_mean", "spectral_centroid_variance",
                                                 "zero_crossing_rate_mean", "zero_crossing_rate_variance"],
                                       x_mapper=shared.flatten)
classical_scaler = StandardScaler()
classical_dataset = to_device(scale(split_dataset(classical_X, classical_y), classical_scaler))

In [ ]:
classical = Collection(
    NaiveModel(input_n=classical_X.shape[1]),
    nn.MSELoss(),
    partial(optim.SGD, lr=0.001, momentum=0.9),
    classical_scaler
)

classical_loss_history = train_model(classical, classical_dataset, 50000)

In [ ]:
loss_history(classical_loss_history)

In [ ]:
test_model(classical_dataset, classical, key="train", sample=3)
test_model(classical_dataset, classical, key="test", sample=3)

In [ ]:
def save_result(c: Collection, df: pd.DataFrame)->pd.DataFrame:
    def _inner(row):
        # ["fire_seeker_gain", "fire_seeker_low", "fire_seeker_mid", "fire_seeker_treble"]]

        a = calculate_classical_features(row)[["root_mean_square_mean", "root_mean_square_variance",
                                                 "spectral_centroid_mean", "spectral_centroid_variance",
                                                 "zero_crossing_rate_mean",
                                                 "zero_crossing_rate_variance"]].to_numpy().reshape(1, -1)
        y_hat = c.model(to_tensor(classical_scaler.transform(a)).to(device))[0]

        row["fire_seeker_gain_pred"] = y_hat[0]
        row["fire_seeker_low_pred"] = y_hat[1]
        row["fire_seeker_mid_pred"] = y_hat[2]
        row["fire_seeker_treble_pred"] = y_hat[3]

        return row

    return df.progress_apply(_inner, axis=1)

Then I dumped the model and scaler for use on the machine with VST installed, so we could experiemnt and visalise what was happening.

In [ ]:
test = save_result(classical, filtered)
def conversion(x: Any):
    if isinstance(x, Path):
        return str(x)
    elif isinstance(x, torch.Tensor):
        return x.tolist()
    else:
        raise Exception(f"Do not know how to convert {type(x)} into something serialisable")

with open("../dump/classical.json", "w") as f:
    test.to_json(f, default_handler=conversion)

Then I dumped the model and scaler for use on the machine with VST installed, so we could experiemnt and visalise what was happening.

In [ ]:
from pickle import dump
with open("../dump/classical.pickel", "wb") as f:
    dump(to_shared_collection(classical), f)

The model predicted all input at maximum when gain is high, which made us think that out lack of audio volume normalisation could be causing issue.

## More features